# Stage 3: Time-Based Validation and Baseline Forecasting

This notebook establishes a strict chronological 16-day holdout and evaluates transparent, direct multi-step baselines. It does not use random splitting, machine-learning models, Kaggle test predictions, or submission generation.

In [1]:
from pathlib import Path
import sys
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "src" / "baselines.py").exists() and (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.baselines import KEYS, evaluate_baselines, load_training_data, rmsle

FIGURE_DIR = ROOT / "reports" / "figures" / "baselines"
REPORT_PATH = ROOT / "reports" / "baseline_results.md"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")

data = load_training_data()
history, validation, predictions, scores = evaluate_baselines(data)
best_names = scores.head(3)["baseline"].tolist()
best_name = best_names[0]
print(f"History: {history['date'].min().date()} to {history['date'].max().date()}")
print(f"Validation: {validation['date'].min().date()} to {validation['date'].max().date()}")
print(f"Validation dates: {validation['date'].nunique()}; rows: {len(validation):,}")
print(f"Stores: {validation['store_nbr'].nunique()}; families: {validation['family'].nunique()}")

History: 2013-01-01 to 2017-07-30
Validation: 2017-07-31 to 2017-08-15
Validation dates: 16; rows: 28,512
Stores: 54; families: 33


## 1. Validation design

The final 16 dates of training form the holdout, matching the actual Kaggle test horizon. Every forecast is made from sales dated before validation begins. Random splitting would place later observations into training while evaluating earlier dates, creating leakage and an unrealistic forecasting task.

![Chronological validation timeline](../reports/figures/baselines/01_validation_timeline.png)

In [2]:
fig, ax = plt.subplots(figsize=(10, 2.4))
ax.hlines(1, history["date"].min(), history["date"].max(), color="#3266a8", linewidth=12, label="Historical training")
ax.hlines(1, validation["date"].min(), validation["date"].max(), color="#dd7f2a", linewidth=12, label="16-day validation")
ax.set(title="Strict Chronological Validation Design", xlabel="Date", yticks=[])
ax.legend(frameon=False, loc="upper left")
fig.tight_layout(); fig.savefig(FIGURE_DIR / "01_validation_timeline.png", dpi=150); plt.close(fig)

## 2. Leakage prevention and metric

All lookup values, means, and weekly patterns are computed from the historical portion ending on 30 July 2017. Actual validation sales never update later forecasts. Predictions are clipped at zero, actual sales are verified nonnegative, and every method uses the same RMSLE implementation:

\[
\mathrm{RMSLE}=\sqrt{rac{1}{n}\sum_i[\log(1+\hat y_i)-\log(1+y_i)]^2}.
\]

Missing group estimates use the same fallback hierarchy: store-family estimate, family estimate, global historical estimate, then zero.

## 3. Baselines compared

1. **Zero forecast:** zero everywhere.
2. **Last observed value:** repeat the final pre-validation value for each store-family.
3. **Recent mean (7 days):** repeat the final seven-day pre-validation mean.
4. **Recent mean (28 days):** repeat the final 28-day pre-validation mean.
5. **Weekday mean (8 weeks):** use the recent store-family mean for the matching weekday.
6. **Repeated final week:** repeat the final complete seven-day pattern across all 16 validation days.

## 4. Overall baseline ranking

![Baseline RMSLE ranking](../reports/figures/baselines/02_baseline_ranking.png)

In [3]:
print(scores.to_string(index=False, formatters={"rmsle": "{:.6f}".format, "mae": "{:.2f}".format}))
fig, ax = plt.subplots(figsize=(8, 4.8))
ordered = scores.sort_values("rmsle", ascending=False)
ax.barh(ordered["baseline"], ordered["rmsle"], color="#3266a8")
ax.set(title="Baseline Validation RMSLE (Lower Is Better)", xlabel="RMSLE", ylabel="Baseline")
for y, value in enumerate(ordered["rmsle"]): ax.text(value + 0.02, y, f"{value:.3f}", va="center")
fig.tight_layout(); fig.savefig(FIGURE_DIR / "02_baseline_ranking.png", dpi=150); plt.close(fig)

 rank               baseline    rmsle    mae                                prediction_rule
    1 Weekday mean (8 weeks) 0.520631  84.10         Series-weekday mean from prior 8 weeks
    2  Recent mean (28 days) 0.521588  97.22       Repeat mean of final 28 historical dates
    3   Recent mean (7 days) 0.524003  98.61        Repeat mean of final 7 historical dates
    4    Repeated final week 0.617040  96.54 Repeat final complete 7-day historical pattern
    5    Last observed value 0.659484 207.51              Repeat final pre-validation value
    6          Zero forecast 4.419500 467.14                                      Predict 0


## 5. Error differences across families, stores, and dates

The next summaries use the best baseline. Ranked plots show only the largest errors rather than creating separate charts for all 33 families or 54 stores.

![Highest family errors](../reports/figures/baselines/03_family_errors.png)

![Highest store errors](../reports/figures/baselines/04_store_errors.png)

![Validation-date errors](../reports/figures/baselines/05_date_errors.png)

In [4]:
def grouped_rmsle(frame, group, prediction_name):
    errors = frame[[group, "actual", prediction_name]].copy()
    errors["squared_log_error"] = (np.log1p(errors[prediction_name]) - np.log1p(errors["actual"])) ** 2
    return errors.groupby(group)["squared_log_error"].mean().pow(0.5).sort_values()


family_errors = grouped_rmsle(predictions, "family", best_name)
store_errors = grouped_rmsle(predictions, "store_nbr", best_name)
date_errors = grouped_rmsle(predictions, "date", best_name)

fig, ax = plt.subplots(figsize=(8, 5))
family_errors.tail(10).plot.barh(ax=ax, color="#b64a4a")
ax.set(title=f"Highest Family RMSLE: {best_name}", xlabel="RMSLE", ylabel="Product family")
fig.tight_layout(); fig.savefig(FIGURE_DIR / "03_family_errors.png", dpi=150); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
store_errors.tail(12).plot.barh(ax=ax, color="#dd7f2a")
ax.set(title=f"Highest Store RMSLE: {best_name}", xlabel="RMSLE", ylabel="Store number")
fig.tight_layout(); fig.savefig(FIGURE_DIR / "04_store_errors.png", dpi=150); plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(date_errors.index, date_errors.values, marker="o", color="#3266a8")
ax.set(title=f"Validation-Date RMSLE: {best_name}", xlabel="Validation date", ylabel="RMSLE")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout(); fig.savefig(FIGURE_DIR / "05_date_errors.png", dpi=150); plt.close(fig)

print("Lowest-error families:", ", ".join(family_errors.head(5).index))
print("Highest-error families:", ", ".join(family_errors.tail(5).index))
print("Highest-error stores:", ", ".join(map(str, store_errors.tail(5).index)))

Lowest-error families: BOOKS, PRODUCE, DAIRY, BREAD/BAKERY, GROCERY I
Highest-error families: BEAUTY, CELEBRATION, GROCERY II, LINGERIE, SCHOOL AND OFFICE SUPPLIES
Highest-error stores: 20, 44, 48, 47, 50


## 6. Representative validation forecasts

Series are selected mechanically from high, median, and lower deciles of pre-validation total sales. This avoids choosing only visually successful examples. Each panel shows 56 historical days, validation actuals, the best forecast, and the 28-day mean as a contrast.

![Representative validation forecasts](../reports/figures/baselines/06_representative_forecasts.png)

In [5]:
series_volume = history.groupby(["store_nbr", "family"])["sales"].sum().sort_values()
positions = [int(0.1 * (len(series_volume) - 1)), int(0.5 * (len(series_volume) - 1)), int(0.9 * (len(series_volume) - 1))]
selected = [series_volume.index[i] for i in positions]
contrast = "Recent mean (28 days)"
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
history_start = validation["date"].min() - pd.Timedelta(days=56)
for ax, (store, family) in zip(axes, selected):
    hist = history.loc[(history["store_nbr"] == store) & (history["family"] == family) & (history["date"] >= history_start)]
    val = predictions.loc[(predictions["store_nbr"] == store) & (predictions["family"] == family)]
    ax.plot(hist["date"], hist["sales"], color="#777777", label="History")
    ax.plot(val["date"], val["actual"], color="black", marker="o", label="Validation actual")
    ax.plot(val["date"], val[best_name], color="#3266a8", linestyle="--", label=best_name)
    ax.plot(val["date"], val[contrast], color="#dd7f2a", linestyle=":", label=contrast)
    ax.set_ylabel("Sales"); ax.set_title(f"Store {store} - {family}", loc="left", fontsize=10)
axes[0].legend(frameon=False, ncol=2, fontsize=8)
axes[-1].set_xlabel("Date")
fig.suptitle("Representative Store-Family Validation Forecasts", y=1.01)
fig.tight_layout(); fig.savefig(FIGURE_DIR / "06_representative_forecasts.png", dpi=150, bbox_inches="tight"); plt.close(fig)
print("Selected series:", selected)

Selected series: [(np.int64(44), 'BOOKS'), (np.int64(44), 'LADIESWEAR'), (np.int64(6), 'DAIRY')]


## 7. Implications and report

The best simple baseline becomes the benchmark that later feature-based approaches must improve under the same time-based split. Weekly seasonality, promotions, special periods, and heterogeneous family scales deserve attention later, but no Stage 4 feature matrix is created here.

In [6]:
score_rows = "\n".join(
    f"| {int(row['rank'])} | {row['baseline']} | {row['rmsle']:.6f} | {row['mae']:.2f} | {row['prediction_rule']} |"
    for _, row in scores.iterrows()
)
report = f"""# Baseline Forecasting Results

## 1. Purpose

Stage 3 establishes a leakage-safe chronological evaluation and transparent benchmarks. It does not train a machine-learning model or forecast the Kaggle test set.

## 2. Validation design

Historical training runs from **{history['date'].min().date()}** through **{history['date'].max().date()}**. Validation uses the final 16 training dates, **{validation['date'].min().date()}** through **{validation['date'].max().date()}**, with **{len(validation):,} rows**, {validation['store_nbr'].nunique()} stores, and {validation['family'].nunique()} families. Random splitting would mix future observations into training and misrepresent the competition forecast setting.

![Chronological validation design](figures/baselines/01_validation_timeline.png)

## 3. Leakage prevention

Every prediction is computed only from sales before validation starts. Validation targets never update later forecasts, centered windows are not used, and the repeated-week method repeats the final complete historical week instead of reading validation lags. Missing estimates fall back from store-family to family, global history, and finally zero.

## 4. Evaluation metric

RMSLE is the primary metric. Predictions are clipped to zero, actual sales are verified nonnegative, and all methods use the same NumPy implementation. MAE is included only as a secondary scale-dependent description.

## 5. Baselines compared

Six methods were evaluated: zero, last value, 7-day mean, 28-day mean, recent 8-week weekday mean, and a leakage-safe repeated final-week pattern.

## 6. Overall results

| Rank | Baseline | RMSLE | MAE | Rule |
|---:|---|---:|---:|---|
{score_rows}

The best method is **{best_name}** with RMSLE **{scores.iloc[0]['rmsle']:.6f}**. The 28-day and 7-day means are close behind. Weekly weekday matching provides a small improvement over these local-level averages, while copying one exact week or one final value is less stable.

![Baseline RMSLE ranking](figures/baselines/02_baseline_ranking.png)

## 7. Error differences across families and stores

The easiest families under the best baseline include {', '.join(family_errors.head(3).index)}. The highest-error families include {', '.join(family_errors.tail(3).index)}. Store errors also vary; stores {', '.join(map(str, store_errors.tail(3).index))} have the largest RMSLE under this baseline. These rankings show where simple local seasonal averages fail, not why they fail.

![Highest family errors](figures/baselines/03_family_errors.png)

## 8. Representative forecast behavior

Mechanically selected low-, medium-, and high-volume series show that a stable aggregate score can coexist with flat forecasts, intermittent targets, and short-lived spikes. Simple averages cannot react to unexpected events within the 16-day horizon.

![Representative validation forecasts](figures/baselines/06_representative_forecasts.png)

## 9. Main findings

- The 8-week weekday mean is the strongest baseline at RMSLE {scores.iloc[0]['rmsle']:.6f}.
- Weekly seasonal matching helps slightly relative to 7- and 28-day constant means.
- The 28-day mean is slightly more stable than the 7-day mean.
- Repeating one exact week is weaker than averaging corresponding weekdays across eight weeks.
- Error differs substantially across families, stores, and validation dates.
- Zero and last-value forecasts are inadequate reference methods for this data.

## 10. Implications for Stage 4

Later work should retain this exact split and beat RMSLE **{scores.iloc[0]['rmsle']:.6f}**. Promotions and calendar information may help, but must be constructed only from information available at forecast time. Family and store heterogeneity should remain visible in evaluation.

## 11. Limitations

This is one recent 16-day holdout and may be affected by unusual dates. Baselines do not adjust for promotions, holidays, trend changes, or special events. The score does not guarantee Kaggle leaderboard performance, and no method is claimed to be universally best.

## 12. Conclusion

The validation framework is chronological, reproducible, and leakage-safe. The recent weekday mean provides a clear benchmark for later stages without introducing a machine-learning model or generating test predictions.
"""
REPORT_PATH.write_text(report, encoding="utf-8")
print(f"Wrote {REPORT_PATH}")
print(f"Saved {len(list(FIGURE_DIR.glob('*.png')))} figures")

Wrote C:\Users\GYDROC\Documents\Codex\2026-07-15\github-plugin-github-openai-curated-remote\work\stage1-repo\reports\baseline_results.md
Saved 6 figures
